In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [3]:
df1 = pd.read_csv(r"C:\Users\ADMIN\Documents\subem\2026\Guvi\Project\BrandVisibility\data\raw\brand_dirty_dataset.csv")

In [4]:
df2 = pd.read_csv(r"C:\Users\ADMIN\Documents\subem\2026\Guvi\Project\BrandVisibility\data\raw\pandas_shopping_results.csv")

In [9]:
print(f"df1 shape: {df1.shape}")
print(f"df2 shape: {df2.shape}")
display(df1.head(3))
display(df2.head(3))

df1 shape: (1320, 7)
df2 shape: (1320, 10)


,keyword,title,price,rating,reviews,platform,delivery
0,office chair ergonomic,HP office chair ergonomic Model 603,36225.0,3.7,3241,croma,2 Days
1,microwave oven convection,Samsung microwave oven convection Model 524 !!!,NaN,4.7,1294,amazon,Free Delivery
2,power bank fast charging,Samsung power bank fast charging Model 402,Not Available,NaN,3070,CROMA,Free Delivery


,keyword,title,price,rating,reviews,platform,position,delivery,link,thumbnail
0,air fryer,Ninja 4-in-1 Air Fryer Pro,$89.99,4.7,1300.0,Best Buy,1,Free delivery by Mon,NaN,https://encrypted-tbn2.gstatic.com/shopping?q=...
1,air fryer,Gourmia Digital Air Fryer 3qt with Guided Cook...,$29.99,3.8,177.0,Target,2,Free delivery on $35+,NaN,https://encrypted-tbn3.gstatic.com/shopping?q=...
2,air fryer,COSORI Dual Blaze Air Fryer,$129.00,4.5,105.0,Cosori,3,Free delivery by Fri,NaN,https://encrypted-tbn3.gstatic.com/shopping?q=...


In [6]:
df1.head(5)


,keyword,title,price,rating,reviews,platform,delivery
0,office chair ergonomic,HP office chair ergonomic Model 603,36225.0,3.7,3241,croma,2 Days
1,microwave oven convection,Samsung microwave oven convection Model 524 !!!,NaN,4.7,1294,amazon,Free Delivery
2,power bank fast charging,Samsung power bank fast charging Model 402,Not Available,NaN,3070,CROMA,Free Delivery
3,microwave oven convection,Dell microwave oven convection Model 877,2507.0,4.7,3446,flipkart,Free Delivery
4,men running shoes,Dell men running shoes Model 536,2176.0,3.8,2031,croma,5 Days


In [7]:
df2.head(5)

,keyword,title,price,rating,reviews,platform,position,delivery,link,thumbnail
0,air fryer,Ninja 4-in-1 Air Fryer Pro,$89.99,4.7,1300.0,Best Buy,1,Free delivery by Mon,NaN,https://encrypted-tbn2.gstatic.com/shopping?q=...
1,air fryer,Gourmia Digital Air Fryer 3qt with Guided Cook...,$29.99,3.8,177.0,Target,2,Free delivery on $35+,NaN,https://encrypted-tbn3.gstatic.com/shopping?q=...
2,air fryer,COSORI Dual Blaze Air Fryer,$129.00,4.5,105.0,Cosori,3,Free delivery by Fri,NaN,https://encrypted-tbn3.gstatic.com/shopping?q=...
3,air fryer,Cuisinart 4.5 Qt Basket Air Fryer,$79.95,4.8,319.0,Cuisinart,4,Free delivery,NaN,https://encrypted-tbn3.gstatic.com/shopping?q=...
4,air fryer,bella PRO 4-qt. TriZone Touchscreen Air Fryer ...,$49.99,4.6,957.0,Best Buy,5,Free delivery by Mon,NaN,https://encrypted-tbn3.gstatic.com/shopping?q=...


In [8]:
print(df1.columns)
print(df2.columns)

Index(['keyword', 'title', 'price', 'rating', 'reviews', 'platform',
       'delivery'],
      dtype='object')
Index(['keyword', 'title', 'price', 'rating', 'reviews', 'platform',
       'position', 'delivery', 'link', 'thumbnail'],
      dtype='object')


In [11]:
df = pd.concat([df1, df2], ignore_index=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2640 entries, 0 to 2639
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   keyword    2640 non-null   object 
 1   title      2640 non-null   object 
 2   price      2514 non-null   object 
 3   rating     2422 non-null   float64
 4   reviews    2553 non-null   object 
 5   platform   2640 non-null   object 
 6   delivery   2119 non-null   object 
 7   position   1320 non-null   float64
 8   link       0 non-null      float64
 9   thumbnail  1320 non-null   object 
dtypes: float64(3), object(7)
memory usage: 206.4+ KB


In [12]:
df = df.drop(columns = 'link')

In [18]:
cat_col = ["keyword", "title", "platform", "delivery"]
for col in cat_col:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({"nan":np.nan,"None":np.nan," ":np.nan,"":np.nan})

for col in ["keyword","platform"]:
    df[col] = df[col].str.lower()
    
    
def handle_price(val):
    if pd.isna(val):
        return np.nan
    cleaned_price = re.sub(r"[^\d.]","",str(val))
    try:
        return float(cleaned_price) if cleaned_price else np.nan
    except ValueError:
        return np.nan
    
df['price_clean'] = df['price'].apply(handle_price)


def handle_reviews(val):
    if pd.isna(val):
        return 0
    text = str(val).lower().replace(",", "").strip()
    match = re.search(r"(\d+\.?\d*)\s*([km]?)", text)
    if not match:
        return 0
    num = float(match.group(1))
    suffix = match.group(2)
    if suffix == "k":
        num *= 1_000
    elif suffix == "m":
        num *= 1_000_000
    return int(num)

df["reviews_clean"] = df["reviews"].apply(handle_reviews)


df.loc[(df['rating'] < 0) | (df['rating'] > 5),"rating"] = np.nan

df = df.sort_values(by="position", na_position="last")
df = df.drop_duplicates(subset=["keyword", "title", "platform"], keep="first").reset_index(drop=True)

print(f"Final cleaned dataset shape: {df.shape}")
display(df[["keyword", "title", "price_clean", "rating", "reviews_clean", "platform", "position"]].head())

Final cleaned dataset shape: (2021, 11)


,keyword,title,price_clean,rating,reviews_clean,platform,position
0,air fryer,Ninja 4-in-1 Air Fryer Pro,89.99,4.7,1300,best buy,1.0
1,grinder,DeWalt Small Angle Grinder DWE4011,59.00,4.4,749,home depot,1.0
2,office chair ergonomic,Kerdom Ergonomic Office Chair,89.99,4.5,24,best buy,1.0
3,head phones,Apple EarPods with Lightning Connector,19.00,4.6,52000,apple,1.0
4,microwave oven convection,Mainstays 3-in-1 Air Fry Convection Microwave ...,104.00,4.4,52,walmart,1.0


In [33]:
print("Total unique keywords:", df["keyword"].nunique())
print(df["keyword"].value_counts())

mapping_product = {
    "microwave oven convection":"microwave_oven",
    "office chair ergonomic" : "office_chair",
    "air fryer india" : "air_fryer",
    "air fryer" : "air_fryer", 
    "men running shoes" : "men_running_shoes",
    "men running shoe" : "men_running_shoes",
    "power bank fast charging":"power_bank",
    "power bank fat charging" : "power_bank",
    "smart phone" : "smart_phone",
    "head phones" : "head_phones",
    "washing machine" :  "washing_machine",
    "bluetooth" : "bluetooth",
    "grinder" :"grinder",
    "chopper" :"chopper",    
}

df['Product'] = df['keyword'].map(mapping_product)


Total unique keywords: 14
keyword
microwave oven convection    330
office chair ergonomic       282
air fryer india              243
men running shoes            240
power bank fast charging     239
power bank fat charging       94
men running shoe              91
air fryer                     83
smart phone                   80
head phones                   74
washing machine               74
bluetooth                     71
grinder                       65
chopper                       55
Name: count, dtype: int64


In [34]:
print("Unmapped rows:", df["Product"].isna().sum())
if df["Product"].isna().sum() > 0:
    print("Missing keywords:", df[df["Product"].isna()]["keyword"].unique())


Unmapped rows: 0


In [35]:
def refine_product(row):
    prod = row["Product"]
    title = str(row["title"]).lower()
    
    if prod == "bluetooth":
        if any(w in title for w in ["earbud", "headphone", "headset", "earphone", "airpod"]):
            return "head_phones"
        elif "speaker" in title:
            return "bluetooth_speaker"
    return prod

df["Product"] = df.apply(refine_product, axis=1)
print(df["Product"].value_counts())


Product
power_bank           333
men_running_shoes    331
microwave_oven       330
air_fryer            326
office_chair         282
head_phones           91
smart_phone           80
washing_machine       74
grinder               65
chopper               55
bluetooth             38
bluetooth_speaker     16
Name: count, dtype: int64


In [36]:
df["Brand"] = (
    df["title"]
    .astype(str)
    .str.strip()
    .str.split()
    .str[0]
    .str.replace(r"[^a-zA-Z0-9\+\&\-]", "", regex=True)
    .str.title()
)

# Check unique brands and top counts
print("Total unique brands:", df["Brand"].nunique())
print("\nTop 20 Brands:\n", df["Brand"].value_counts().head(20))

# Preview alongside Product and title
display(df[["Product", "Brand", "title", "price_clean", "platform"]].head(10))


Total unique brands: 298

Top 20 Brands:
 Brand
Lg           168
Samsung      165
Hp           156
Nike         155
Philips      138
Puma         138
Boat         134
Sony         126
Dell         120
Apple         20
Cuisinart     17
Ninja         17
Ge            16
Anker         16
Jbl           16
Adidas        13
Motorola      11
Whirlpool     11
Dewalt        10
Google        10
Name: count, dtype: int64


,Product,Brand,title,price_clean,platform
0,air_fryer,Ninja,Ninja 4-in-1 Air Fryer Pro,89.99,best buy
1,grinder,Dewalt,DeWalt Small Angle Grinder DWE4011,59.00,home depot
2,office_chair,Kerdom,Kerdom Ergonomic Office Chair,89.99,best buy
3,head_phones,Apple,Apple EarPods with Lightning Connector,19.00,apple
4,microwave_oven,Mainstays,Mainstays 3-in-1 Air Fry Convection Microwave ...,104.00,walmart
5,chopper,Kitchen,Kitchen Collection Food Chopper 72410KC,14.99,target
6,head_phones,Anker,Anker Soundcore Q20i Hybrid Active Noise Cance...,44.99,best buy
7,air_fryer,Gourmia,Gourmia Digital Air Fryer 3qt with Guided Cook...,29.99,target
8,bluetooth,Bluetooth,Bluetooth Wireless Audio Receiver Adapter,5.00,five below
9,men_running_shoes,Nike,Nike Men's Revolution 8 Low-Top Running Shoes,69.98,famous footwear


In [38]:
df = df.drop(columns = ['price','reviews'])

In [40]:
df['delivery'].value_counts()

delivery
Free Delivery               429
2 Days                      399
5 Days                      386
Free delivery               302
Free delivery by Fri         44
Free delivery by Mon         38
Free delivery by Thu         32
Free delivery by Wed         24
Free delivery on $35+        20
Free delivery by Tue         10
Free next-day delivery        9
Free delivery on $100+        5
Free delivery on $89+         2
Free delivery on $99+         2
Free delivery on $75+         2
Free delivery on $69+         1
Free delivery on $399+        1
Free delivery on $65+         1
Free delivery on $50+         1
Free delivery on $1,299+      1
Free delivery on $150+        1
Free delivery on $79+         1
Free delivery on $250+        1
Name: count, dtype: int64

In [41]:
def categorize_delivery(val):
    if pd.isna(val) or str(val).strip().lower() in ["", "nan", "none", "not specified"]:
        return "Not Specified"
    
    text = str(val).strip().lower()
    
    # Conditional free shipping (requires minimum order amount like "on $35+")
    if "free" in text and "on $" in text:
        return "Conditional Free Delivery"
    
    # Unconditional free shipping ("Free Delivery", "Free delivery by Fri", "Free next-day delivery")
    if "free" in text:
        return "Free Delivery"
    
    # Fast shipping ("2 Days")
    if "2 days" in text or "next-day" in text:
        return "Fast Delivery (2 Days)"
    
    # Standard shipping ("5 Days")
    if "5 days" in text:
        return "Standard Delivery (5 Days)"
    
    return "Other"

df["delivery_category"] = df["delivery"].apply(categorize_delivery)

# Binary flag: 1 if unconditional Free Delivery, 0 otherwise
df["is_free_delivery"] = (df["delivery_category"] == "Free Delivery").astype(int)

print(df["delivery_category"].value_counts())


delivery_category
Free Delivery                 888
Fast Delivery (2 Days)        399
Standard Delivery (5 Days)    386
Not Specified                 309
Conditional Free Delivery      39
Name: count, dtype: int64


In [42]:
df.head()

,keyword,title,rating,platform,delivery,position,thumbnail,price_clean,reviews_clean,Product,Brand,delivery_category,is_free_delivery
0,air fryer,Ninja 4-in-1 Air Fryer Pro,4.7,best buy,Free delivery by Mon,1.0,https://encrypted-tbn2.gstatic.com/shopping?q=...,89.99,1300,air_fryer,Ninja,Free Delivery,1
1,grinder,DeWalt Small Angle Grinder DWE4011,4.4,home depot,Free delivery by Mon,1.0,https://encrypted-tbn1.gstatic.com/shopping?q=...,59.00,749,grinder,Dewalt,Free Delivery,1
2,office chair ergonomic,Kerdom Ergonomic Office Chair,4.5,best buy,NaN,1.0,https://encrypted-tbn3.gstatic.com/shopping?q=...,89.99,24,office_chair,Kerdom,Not Specified,0
3,head phones,Apple EarPods with Lightning Connector,4.6,apple,Free next-day delivery,1.0,https://encrypted-tbn1.gstatic.com/shopping?q=...,19.00,52000,head_phones,Apple,Free Delivery,1
4,microwave oven convection,Mainstays 3-in-1 Air Fry Convection Microwave ...,4.4,walmart,Free delivery,1.0,https://encrypted-tbn3.gstatic.com/shopping?q=...,104.00,52,microwave_oven,Mainstays,Free Delivery,1


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2021 entries, 0 to 2020
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   keyword            2021 non-null   object 
 1   title              2021 non-null   object 
 2   rating             1839 non-null   float64
 3   platform           2021 non-null   object 
 4   delivery           1712 non-null   object 
 5   position           807 non-null    float64
 6   thumbnail          807 non-null    object 
 7   price_clean        1845 non-null   float64
 8   reviews_clean      2021 non-null   int64  
 9   Product            2021 non-null   object 
 10  Brand              2021 non-null   object 
 11  delivery_category  2021 non-null   object 
 12  is_free_delivery   2021 non-null   int64  
dtypes: float64(3), int64(2), object(8)
memory usage: 205.4+ KB


In [44]:
col_to_drop = ['keyword','delivery','thumbnail']

df = df.drop(columns = [c for c in col_to_drop if c in df.columns])

df = df.rename(columns={
    "price_clean": "price",
    "reviews_clean": "review"
})

In [45]:
print(df.isna().sum())


title                   0
rating                182
platform                0
position             1214
price                 176
review                  0
Product                 0
Brand                   0
delivery_category       0
is_free_delivery        0
dtype: int64


In [46]:
df["price"] = (
    df["price"]
    .fillna(df.groupby("Product")["price"].transform("median"))
    .fillna(df["price"].median())
)

In [47]:
df["rating"] = (
    df["rating"]
    .fillna(df.groupby("Product")["rating"].transform("median"))
    .fillna(df["rating"].median())
)

In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2021 entries, 0 to 2020
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              2021 non-null   object 
 1   rating             2021 non-null   float64
 2   platform           2021 non-null   object 
 3   position           807 non-null    float64
 4   price              2021 non-null   float64
 5   review             2021 non-null   int64  
 6   Product            2021 non-null   object 
 7   Brand              2021 non-null   object 
 8   delivery_category  2021 non-null   object 
 9   is_free_delivery   2021 non-null   int64  
dtypes: float64(3), int64(2), object(5)
memory usage: 158.0+ KB


In [49]:
df.to_csv("../data/Processed/cleaned_brand_visibility.csv", index=False)
print("Successfully saved cleaned dataset! Shape:", df.shape)


Successfully saved cleaned dataset! Shape: (2021, 10)


Feature Engineering

In [52]:
import re
import numpy as np
import pandas as pd

df = pd.read_csv("../data/Processed/cleaned_brand_visibility.csv")


display(df[["Product", "Brand", "title", "price", "position"]].head())


,Product,Brand,title,price,position
0,air_fryer,Ninja,Ninja 4-in-1 Air Fryer Pro,89.99,1.0
1,grinder,Dewalt,DeWalt Small Angle Grinder DWE4011,59.00,1.0
2,office_chair,Kerdom,Kerdom Ergonomic Office Chair,89.99,1.0
3,head_phones,Apple,Apple EarPods with Lightning Connector,19.00,1.0
4,microwave_oven,Mainstays,Mainstays 3-in-1 Air Fry Convection Microwave ...,104.00,1.0


In [53]:
df["rank_visibility_score"] = np.where(
    df["position"].notna(),
    (100 / np.log2(df["position"] + 1)).round(2),
    0.0
)


In [54]:
# Category-relative Price Tiers (Budget / Mid-Range / Premium per Product)
df["price_range"] = df.groupby("Product")["price"].transform(
    lambda x: pd.qcut(x, q=3, labels=["Budget", "Mid-Range", "Premium"], duplicates="drop")
)

# Also calculate price relative to category average (e.g., 1.20 = 20% more expensive than category avg)
cat_avg_price = df.groupby("Product")["price"].transform("mean")
df["price_vs_category_avg"] = (df["price"] / cat_avg_price).round(2)

print(df["price_range"].value_counts())
display(df[["Product", "Brand", "price", "price_range", "price_vs_category_avg"]].head(10))


price_range
Budget       678
Premium      672
Mid-Range    671
Name: count, dtype: int64


,Product,Brand,price,price_range,price_vs_category_avg
0,air_fryer,Ninja,89.99,Budget,0.00
1,grinder,Dewalt,59.00,Mid-Range,0.40
2,office_chair,Kerdom,89.99,Budget,0.00
3,head_phones,Apple,19.00,Budget,0.06
4,microwave_oven,Mainstays,104.00,Budget,0.00
5,chopper,Kitchen,14.99,Budget,0.11
6,head_phones,Anker,44.99,Mid-Range,0.14
7,air_fryer,Gourmia,29.99,Budget,0.00
8,bluetooth,Bluetooth,5.00,Budget,0.21
9,men_running_shoes,Nike,69.98,Budget,0.00


In [55]:
# Method A: Extract explicit "% off" from product title if present
df["title_discount_pct"] = (
    df["title"]
    .str.extract(r"(\d+)\s*%\s*off", flags=re.IGNORECASE)
    .astype(float)
    .fillna(0.0)
)

# Method B: Category Benchmark Discount (% below category 75th percentile price)
cat_benchmark_price = df.groupby("Product")["price"].transform(lambda x: x.quantile(0.75))
df["category_discount_pct"] = np.where(
    df["price"] < cat_benchmark_price,
    ((cat_benchmark_price - df["price"]) / cat_benchmark_price * 100).round(2),
    0.0
)

print("Products with explicit '% off' in title:", (df["title_discount_pct"] > 0).sum())
print("Products priced below category benchmark:", (df["category_discount_pct"] > 0).sum())
display(df[["Product", "Brand", "price", "title_discount_pct", "category_discount_pct"]].head(10))


Products with explicit '% off' in title: 0
Products priced below category benchmark: 1510


,Product,Brand,price,title_discount_pct,category_discount_pct
0,air_fryer,Ninja,89.99,0.0,99.69
1,grinder,Dewalt,59.00,0.0,64.46
2,office_chair,Kerdom,89.99,0.0,99.74
3,head_phones,Apple,19.00,0.0,91.42
4,microwave_oven,Mainstays,104.00,0.0,99.65
5,chopper,Kitchen,14.99,0.0,71.44
6,head_phones,Anker,44.99,0.0,79.69
7,air_fryer,Gourmia,29.99,0.0,99.90
8,bluetooth,Bluetooth,5.00,0.0,85.50
9,men_running_shoes,Nike,69.98,0.0,99.74


In [56]:
df.to_csv("../data/Processed/featured_brand_visibility.csv", index=False)